# MRRpy in five minutes: a first run with nothing to prepare

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SauravBhattarai19/MRRpy/blob/main/notebooks/quickstart_demo.ipynb)

This notebook runs a small sample basin from start to finish. It needs no data and no account. Run the cells from top to bottom.

1. Run the sample basin.
2. Look at the result.
3. Change a setting and run again.
4. Compare several settings in one call.
5. Try a real basin, with a DEM downloaded for you.

## 0. Install

On Colab, or anywhere MRRpy is not installed yet, remove the `#` and run this cell once.

In [ ]:
# %pip install MRRpy

## 1. Run the sample basin

A made-up valley of about 70 km², and a storm of 25 mm/h for two hours. Nothing is downloaded.

In [ ]:
import MRRpy

out = MRRpy.demo("mrrpy_demo")

## 2. Look at the result

`out` holds the paths of every file the run wrote, and a few shortcuts.

In [ ]:
print(out.summary())
out.hydrograph.head()

In [ ]:
fig, ax = out.plot()

In [ ]:
fig, ax = MRRpy.plot_watershed(out)

## 3. Change a setting and run again

The settings of the run are in `mrrpy_demo/run.yaml`. Here the ground soaks up part of the rain (SCS Curve Number 70) instead of shedding all of it.

`MRRpy.explain("RUNOFF_SOURCE")` explains any setting.

In [ ]:
from MRRpy import Config, run_pipeline

cfg = Config.from_file("mrrpy_demo/run.yaml")
cfg.RUNOFF_SOURCE = "scs_cn"
cfg.RUNOFF_CN_SOURCE = "scalar"
cfg.RUNOFF_CN = 70
cfg.OUTPUT_DIR = "mrrpy_demo/results_cn70"

out_cn = run_pipeline(cfg, quiet=True)
print(out_cn.summary())

In [ ]:
fig, ax = out.plot(label="all rain runs off", annotate_peak=False)
out_cn.plot(ax=ax, label="curve number 70", annotate_peak=False)

## 4. Compare several settings in one call

`run_many` processes the terrain once, runs each variation in its own folder and returns one table.

In [ ]:
table = MRRpy.run_many(cfg, {
    "cn60": {"RUNOFF_CN": 60},
    "cn70": {"RUNOFF_CN": 70},
    "cn85": {"RUNOFF_CN": 85},
}, output_dir="mrrpy_demo/curve_numbers")
table[["name", "status", "peak_m3s", "peak_time_hr", "runoff_ratio"]]

## 5. A real basin, with no data of your own

Give an area and an outlet. MRRpy downloads the Copernicus DEM (open data, no account), finds the basin and routes the storm. This cell needs the internet.

To use your own place, change the box (west, south, east, north) and the outlet (latitude, longitude).

In [ ]:
real = Config(
    DEM_BOUNDS_WGS84=(85.48, 27.52, 85.66, 27.68),
    DEM_SOURCE="glo90_open",
    OUTPUT_POINT=(27.575, 85.555),
    OUTPUT_DIR="mrrpy_demo/real_basin",
    RAIN_INTENSITY_MM_HR=25.0, RAIN_DURATION_HOURS=2.0,
    TOTAL_SIMULATION_TIME_HOURS=10.0,
)
real.validate()
out_real = run_pipeline(real, quiet=True)
print(out_real.summary())

In [ ]:
fig, ax = MRRpy.plot_watershed(out_real)

## Next

- `MRRpy.ConfigForm()` opens a step-by-step form for your own basin (notebook `configure_and_run.ipynb`).
- Real storms: satellite rain, weather-model rain and forecasts are in the [user manual](https://mrrpy.readthedocs.io/en/latest/manual/precipitation/).
- Flood depth maps for a town: notebook `flood_maps_2d.ipynb`.